# Task 3 (v3): Stable Training Strategy

Key strategy:
- Phase 1: Frozen encoder, train decoder only (stable convergence)
- Phase 2: Unfreeze ONLY last 3 blocks with very low LR
- Heavy online augmentation
- Dropout + SpatialDropout for regularization
- Longer patience to avoid premature stopping

In [ ]:
import numpy as np
import cv2
import keras
import tensorflow as tf
from keras import layers, Model
from keras.applications import MobileNetV2
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import time

print(f'TensorFlow: {tf.__version__}, Keras: {keras.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

IMG_SIZE = 256
BATCH_SIZE = 8  # smaller batch = more updates per epoch = better generalization
SEED = 42
np.random.seed(SEED)

In [ ]:
X_train = np.load('X_train_aug.npy')
y_train = np.load('y_train_aug.npy')
X_val = np.load('X_val.npy')
y_val = np.load('y_val.npy')

# Use only originals for online augmentation
n_orig = len(X_train) // 2
X_train_orig = X_train[:n_orig]
y_train_orig = y_train[:n_orig]

print(f'Train: {X_train_orig.shape}, Val: {X_val.shape}')

## Online Augmentation Generator

In [ ]:
class DataGenerator(keras.utils.Sequence):
    def __init__(self, images, masks, batch_size=8, augment=True, **kwargs):
        super().__init__(**kwargs)
        self.images = images
        self.masks = masks
        self.batch_size = batch_size
        self.augment = augment
        self.indices = np.arange(len(images))
        if augment:
            np.random.shuffle(self.indices)
    
    def __len__(self):
        return int(np.ceil(len(self.images) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        imgs = self.images[batch_idx].copy()
        msks = self.masks[batch_idx].copy()
        
        if self.augment:
            for i in range(len(imgs)):
                imgs[i], msks[i] = self._augment(imgs[i], msks[i])
        
        return imgs, msks
    
    def _augment(self, img, msk):
        # Horizontal flip (50%)
        if np.random.random() > 0.5:
            img = np.fliplr(img)
            msk = np.fliplr(msk)
        
        # Rotation ±15 degrees (50%)
        if np.random.random() > 0.5:
            angle = np.random.uniform(-15, 15)
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
            img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
            m2d = msk[:,:,0]
            m2d = cv2.warpAffine(m2d, M, (w, h), flags=cv2.INTER_NEAREST)
            msk = m2d[:,:,np.newaxis]
        
        # Brightness (50%)
        if np.random.random() > 0.5:
            img = np.clip(img * np.random.uniform(0.7, 1.3), 0, 1)
        
        # Gaussian blur (30%)
        if np.random.random() > 0.7:
            ksize = np.random.choice([3, 5])
            img = cv2.GaussianBlur(img, (ksize, ksize), 0)
        
        return img.astype(np.float32), msk.astype(np.float32)
    
    def on_epoch_end(self):
        if self.augment:
            np.random.shuffle(self.indices)


train_gen = DataGenerator(X_train_orig, y_train_orig, BATCH_SIZE, augment=True)
val_gen = DataGenerator(X_val, y_val, BATCH_SIZE, augment=False)
print(f'Train batches: {len(train_gen)}, Val batches: {len(val_gen)}')

## Loss & Metrics

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = keras.ops.cast(keras.ops.reshape(y_true, [-1]), 'float32')
    y_pred_f = keras.ops.cast(keras.ops.reshape(y_pred, [-1]), 'float32')
    intersection = keras.ops.sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (keras.ops.sum(y_true_f) + keras.ops.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = keras.ops.mean(keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)

def iou_metric(y_true, y_pred, smooth=1e-6):
    y_true_f = keras.ops.cast(keras.ops.reshape(y_true, [-1]), 'float32')
    y_pred_f = keras.ops.cast(keras.ops.reshape(y_pred > 0.5, [-1]), 'float32')
    intersection = keras.ops.sum(y_true_f * y_pred_f)
    union = keras.ops.sum(y_true_f) + keras.ops.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

## Build U-Net

In [ ]:
def build_unet(input_shape=(256, 256, 3)):
    base = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    
    skip_names = [
        'block_1_expand_relu',
        'block_3_expand_relu',
        'block_6_expand_relu',
        'block_13_expand_relu',
        'block_16_project',
    ]
    skip_outs = [base.get_layer(n).output for n in skip_names]
    encoder = Model(inputs=base.input, outputs=skip_outs, name='encoder')
    encoder.trainable = False  # Fully frozen for Phase 1
    
    inputs = layers.Input(shape=input_shape)
    s1, s2, s3, s4, bottleneck = encoder(inputs, training=False)
    
    def up_block(x, skip, filters):
        x = layers.UpSampling2D(2)(x)
        x = layers.Concatenate()([x, skip])
        x = layers.SeparableConv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.SpatialDropout2D(0.2)(x)
        x = layers.SeparableConv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        return x
    
    x = up_block(bottleneck, s4, 256)
    x = up_block(x, s3, 128)
    x = up_block(x, s2, 64)
    x = up_block(x, s1, 32)
    
    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(16, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(x)
    
    return Model(inputs, outputs, name='UNet_MobileNetV2'), encoder


model, encoder = build_unet()
print(f'Total params: {model.count_params():,}')
print(f'Trainable: {sum(np.prod(w.shape) for w in model.trainable_weights):,}')

## Phase 1: Train Decoder (Frozen Encoder)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=bce_dice_loss,
    metrics=[dice_coefficient, iou_metric, 'accuracy']
)

cb1 = [
    ModelCheckpoint('best_phase1.keras', monitor='val_dice_coefficient', mode='max',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_dice_coefficient', mode='max', patience=20,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_dice_coefficient', mode='max', factor=0.5,
                      patience=8, min_lr=1e-6, verbose=1)
]

print('Phase 1: Frozen encoder, training decoder...')
t0 = time.time()

h1 = model.fit(train_gen, validation_data=val_gen, epochs=80, callbacks=cb1)

t1 = time.time() - t0
print(f'Phase 1 time: {t1/60:.1f} min')

In [ ]:
# Check Phase 1 results
p1_val = model.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)
print('Phase 1 Validation:')
for n, v in zip(model.metrics_names, p1_val):
    print(f'  {n}: {v:.4f}')

## Phase 2: Fine-tune Last 3 Encoder Blocks

In [ ]:
# Get the encoder from inside the model
encoder = model.get_layer('encoder')

# Unfreeze only blocks 14, 15, 16
encoder.trainable = True
for layer in encoder.layers:
    name = layer.name
    block_num = -1
    if 'block_' in name:
        try:
            block_num = int(name.split('block_')[1].split('_')[0])
        except:
            pass
    # Only unfreeze blocks 14-16
    if block_num >= 14:
        layer.trainable = True
    else:
        layer.trainable = False

print(f'Trainable after unfreeze: {sum(np.prod(w.shape) for w in model.trainable_weights):,}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # Very low LR
    loss=bce_dice_loss,
    metrics=[dice_coefficient, iou_metric, 'accuracy']
)

cb2 = [
    ModelCheckpoint('best_phase2.keras', monitor='val_dice_coefficient', mode='max',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_dice_coefficient', mode='max', patience=15,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_dice_coefficient', mode='max', factor=0.5,
                      patience=7, min_lr=1e-7, verbose=1)
]

print('Phase 2: Fine-tuning blocks 14-16...')
t0 = time.time()

h2 = model.fit(train_gen, validation_data=val_gen, epochs=40, callbacks=cb2)

t2 = time.time() - t0
print(f'Phase 2 time: {t2/60:.1f} min')

## Save Model

In [ ]:
model.save_weights('unet_final.weights.h5')
model.save('unet_final.keras')
print('Saved model weights and full model.')

## Training History

In [ ]:
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined, len(h1.history['loss'])

hist, p1_len = combine_histories(h1, h2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, train_key, val_key, title, target in [
    (axes[0], 'loss', 'val_loss', 'Loss', None),
    (axes[1], 'dice_coefficient', 'val_dice_coefficient', 'Dice Coefficient', 0.92),
    (axes[2], 'iou_metric', 'val_iou_metric', 'IoU', 0.88),
]:
    ax.plot(hist[train_key], label='Train', color='steelblue')
    ax.plot(hist[val_key], label='Val', color='coral')
    ax.axvline(x=p1_len, color='gray', linestyle='--', alpha=0.5, label='Fine-tune')
    if target:
        ax.axhline(y=target, color='green', linestyle='--', alpha=0.5, label=f'Target {target}')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history_v3.png', dpi=150, bbox_inches='tight')
plt.show()

## Validation Predictions

In [ ]:
idx = np.random.choice(len(X_val), 6, replace=False)
preds = model.predict(X_val[idx])
preds_bin = (preds > 0.5).astype(np.float32)

fig, axes = plt.subplots(6, 4, figsize=(16, 24))
titles = ['Image', 'Ground Truth', 'Predicted (raw)', 'Predicted (binary)']

for i in range(6):
    axes[i,0].imshow(X_val[idx[i]])
    axes[i,1].imshow(y_val[idx[i],:,:,0], cmap='gray')
    axes[i,2].imshow(preds[i,:,:,0], cmap='gray')
    axes[i,3].imshow(preds_bin[i,:,:,0], cmap='gray')
    for j in range(4):
        axes[i,j].axis('off')
        if i == 0:
            axes[i,j].set_title(titles[j])

plt.tight_layout()
plt.savefig('predictions_v3.png', dpi=150, bbox_inches='tight')
plt.show()

## Final Metrics & Inference Speed

In [ ]:
results = model.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)
print('=== Final Validation Metrics ===')
for n, v in zip(model.metrics_names, results):
    print(f'  {n}: {v:.4f}')

# Inference speed
times = []
for _ in range(30):
    t = time.time()
    _ = model.predict(X_val[:1], verbose=0)
    times.append((time.time() - t) * 1000)

print(f'\n  Inference: {np.mean(times[5:]):.1f} ms/image')
print(f'  Total training: {(t1+t2)/60:.1f} min')
print('\nTask 3 Complete!')